# XGBoost Hyperparameter Tuning (Optuna)

**Objective**: Minimize 10-day autoregressive MSPE  
**Train**: 2015 | **Validation**: 2016  
**Output**: `best_xgb_params.json`

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from calendar import monthrange
from xgboost import XGBRegressor
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

TARGET = "YF_Price_Crude Oil (WTI)"
LOOKBACK = 10
FORECAST_HORIZON = 10

## Load Data & Feature Engineering
Same pipeline as baseline notebook.

In [ ]:
df = pd.read_csv("combined_commodity_data.csv", parse_dates=["date"], index_col="date")
raw_cols = df.columns.tolist()

# Rolling window averages
for window in [3, 5, 10]:
    rolling = df[raw_cols].rolling(window=window, min_periods=1).mean()
    rolling.columns = [f"{c}_roll{window}" for c in raw_cols]
    df = pd.concat([df, rolling], axis=1)

# Cyclic time features
dow = df.index.dayofweek
dom = df.index.day
month = df.index.month
days_in_month = df.index.to_series().apply(lambda d: monthrange(d.year, d.month)[1])

df["time_dow_sin"] = np.sin(2 * np.pi * dow / 7)
df["time_dow_cos"] = np.cos(2 * np.pi * dow / 7)
df["time_dom_sin"] = np.sin(2 * np.pi * (dom - 1) / days_in_month)
df["time_dom_cos"] = np.cos(2 * np.pi * (dom - 1) / days_in_month)
df["time_month_sin"] = np.sin(2 * np.pi * (month - 1) / 12)
df["time_month_cos"] = np.cos(2 * np.pi * (month - 1) / 12)

time_cols = [c for c in df.columns if c.startswith("time_")]
all_feature_cols = raw_cols + [c for c in df.columns if "_roll" in c]

# Build lookback features
def build_lookback_features(df, raw_cols, all_feature_cols, time_cols, lookback=10):
    rolling_cols = [c for c in all_feature_cols if "_roll" in c]
    n = len(df)
    raw_arr = df[raw_cols].values
    roll_arr = df[rolling_cols].values
    time_arr = df[time_cols].values
    target_arr = df[TARGET].values
    idx = df.index
    X_rows, y_vals, dates = [], [], []
    for t in range(lookback, n):
        raw_flat = raw_arr[t - lookback:t][::-1].flatten()
        roll_at_t1 = roll_arr[t - 1]
        time_at_t = time_arr[t]
        X_rows.append(np.concatenate([raw_flat, roll_at_t1, time_at_t]))
        y_vals.append(target_arr[t])
        dates.append(idx[t])
    return np.array(X_rows), np.array(y_vals), dates

X_all, y_all, dates_all = build_lookback_features(df, raw_cols, all_feature_cols, time_cols, LOOKBACK)
dates_arr = pd.DatetimeIndex(dates_all)

# Train: 2015, Val: 2016
train_mask = (dates_arr >= "2015-01-01") & (dates_arr < "2016-01-01")
val_mask = (dates_arr >= "2016-01-01") & (dates_arr < "2017-01-01")

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]

print(f"Train: {X_train.shape[0]} samples | Val: {X_val.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")

## Autoregressive Evaluation Function

In [ ]:
df_raw = df[raw_cols].copy()

def build_single_feature_row(buffer_raw, target_date, raw_cols, lookback=10):
    """Build one feature row from a buffer for autoregressive prediction."""
    raw_flat = buffer_raw[raw_cols].values[-lookback:][::-1].flatten()
    roll_vals = []
    for window in [3, 5, 10]:
        rolled = buffer_raw[raw_cols].rolling(window=window, min_periods=1).mean().iloc[-1].values
        roll_vals.append(rolled)
    roll_flat = np.concatenate(roll_vals)
    dow = target_date.dayofweek
    dom = target_date.day
    month = target_date.month
    dim = monthrange(target_date.year, target_date.month)[1]
    time_feats = np.array([
        np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7),
        np.sin(2 * np.pi * (dom - 1) / dim), np.cos(2 * np.pi * (dom - 1) / dim),
        np.sin(2 * np.pi * (month - 1) / 12), np.cos(2 * np.pi * (month - 1) / 12),
    ])
    return np.concatenate([raw_flat, roll_flat, time_feats]).reshape(1, -1)


def evaluate_ar_mspe(model, df_raw, raw_cols, val_start, val_end, lookback=10, horizon=10):
    """Compute average 10-day autoregressive MSPE over non-overlapping windows."""
    val_dates = df_raw.loc[val_start:val_end].index
    window_mspe_list = []
    origin_idx = 0
    while origin_idx + horizon <= len(val_dates):
        forecast_dates = val_dates[origin_idx:origin_idx + horizon]
        origin_pos = df_raw.index.get_loc(forecast_dates[0])
        buffer = df_raw.iloc[origin_pos - lookback:origin_pos].copy()
        preds, actuals = [], []
        for step in range(horizon):
            target_date = forecast_dates[step]
            actual_val = df_raw.loc[target_date, TARGET]
            row = build_single_feature_row(buffer, target_date, raw_cols, lookback)
            pred = model.predict(row)[0]
            preds.append(pred)
            actuals.append(actual_val)
            new_row = buffer.iloc[-1].copy()
            new_row[TARGET] = pred
            new_row.name = target_date
            buffer = pd.concat([buffer, new_row.to_frame().T])
            buffer = buffer.iloc[-lookback:]
        preds, actuals = np.array(preds), np.array(actuals)
        window_mspe_list.append(((preds - actuals) / actuals).clip(-10, 10) ** 2)
        origin_idx += horizon
    if not window_mspe_list:
        return float("inf")
    return np.mean([w.mean() for w in window_mspe_list])

print("Evaluation function ready.")

## Optuna Study

**Tuned hyperparameters:**
- `n_estimators` (100-1000) with early stopping
- `max_depth` (3-10)
- `learning_rate` (0.005-0.3)
- `subsample` (0.5-1.0)
- `colsample_bytree` (0.3-1.0)
- `min_child_weight` (1-20)
- `reg_alpha` (L1, 1e-8 to 10)
- `reg_lambda` (L2, 1e-8 to 10)
- `gamma` (min split loss, 0-5)

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "random_state": 42,
        "n_jobs": -1,
        "early_stopping_rounds": 50,
    }

    model = XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    mspe = evaluate_ar_mspe(model, df_raw, raw_cols, "2016-01-01", "2016-12-31")
    return mspe


study = optuna.create_study(direction="minimize", study_name="xgb_wti_tuning")
study.optimize(objective, n_trials=60, show_progress_bar=True)

print(f"\nBest MSPE: {study.best_value:.6f}  (RMSPE: {np.sqrt(study.best_value)*100:.2f}%)")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

## Save Best Hyperparameters

In [ ]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["early_stopping_rounds"] = 50

output = {
    "best_params": best_params,
    "best_mspe": study.best_value,
    "best_rmspe_pct": np.sqrt(study.best_value) * 100,
    "train_period": "2015-01-01 to 2015-12-31",
    "val_period": "2016-01-01 to 2016-12-31",
    "n_trials": len(study.trials),
}

with open("best_xgb_params.json", "w") as f:
    json.dump(output, f, indent=2)

print("Saved to best_xgb_params.json")
print(json.dumps(output, indent=2))

## Optimization Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Optimization history
ax = axes[0]
trial_values = [t.value for t in study.trials if t.value is not None]
best_so_far = np.minimum.accumulate(trial_values)
ax.scatter(range(len(trial_values)), trial_values, alpha=0.4, s=15, label="Trial MSPE")
ax.plot(best_so_far, color="red", linewidth=2, label="Best so far")
ax.set_xlabel("Trial")
ax.set_ylabel("MSPE")
ax.set_title("Optimization History")
ax.legend()

# 2. Parameter importances (approximate via correlation with objective)
ax = axes[1]
param_names = list(study.best_params.keys())
importances = optuna.importance.get_param_importances(study)
names = list(importances.keys())
vals = list(importances.values())
ax.barh(range(len(names)), vals[::-1], color="steelblue")
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names[::-1], fontsize=9)
ax.set_xlabel("Importance")
ax.set_title("Hyperparameter Importance")

# 3. Top params scatter: learning_rate vs max_depth colored by MSPE
ax = axes[2]
lrs = [t.params.get("learning_rate") for t in study.trials if t.value is not None]
depths = [t.params.get("max_depth") for t in study.trials if t.value is not None]
sc = ax.scatter(lrs, depths, c=trial_values, cmap="RdYlGn_r", s=30, alpha=0.7)
ax.set_xlabel("learning_rate")
ax.set_ylabel("max_depth")
ax.set_xscale("log")
ax.set_title("learning_rate vs max_depth (color=MSPE)")
plt.colorbar(sc, ax=ax, label="MSPE")

plt.tight_layout()
plt.show()